### Build a Question Answering application over a Graph Database

In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_groq import ChatGroq

In [2]:
# ============================================================
# 2. LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")

In [3]:
# ============================================================
# 3. VALIDATE ENVIRONMENT VARIABLES
# ============================================================

required_variables = {
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_USERNAME": NEO4J_USERNAME,
    "NEO4J_PASSWORD": NEO4J_PASSWORD,
    "NEO4J_DATABASE": NEO4J_DATABASE,
    "GROQ_API_KEY": GROQ_API_KEY,
    "GROQ_MODEL": GROQ_MODEL,
}

missing_variables = [
    name
    for name, value in required_variables.items()
    if not value
]

if missing_variables:
    raise ValueError(
        f"Missing environment variables: {', '.join(missing_variables)}"
    )

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [4]:
# ============================================================
# 4. CONNECT TO NEO4J
# ============================================================

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

print("Neo4j connection successful!")

Neo4j connection successful!


In [5]:
# ============================================================
# 5. CHECK NEO4J DATABASE
# ============================================================

databases = graph.query("SHOW DATABASES")

for database in databases:
    print(database)

{'name': '1e3a6720', 'type': 'standard', 'aliases': [], 'access': 'read-write', 'address': 'p-mt-2e58c6e6c448-2-0164.production-orch-0703.neo4j.io:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'default': False, 'home': True, 'constituents': []}
{'name': 'system', 'type': 'system', 'aliases': [], 'access': 'read-write', 'address': 'p-mt-2e58c6e6c448-2-0164.production-orch-0703.neo4j.io:7687', 'role': 'primary', 'writer': False, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'default': False, 'home': False, 'constituents': []}
{'name': 'system', 'type': 'system', 'aliases': [], 'access': 'read-write', 'address': 'p-mt-2e58c6e6c448-2-0163.production-orch-0703.neo4j.io:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'default': False, 'home': False, 'constituents': []}
{'name': 'system', 'type': 'system', 'aliases': [],

In [6]:
# ============================================================
# 6. MOVIE DATASET QUERY
# ============================================================

movie_query = """
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv'
AS row

MERGE (m:Movie {id: row.movieId})

SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

FOREACH (director IN split(row.director, '|') |
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED]->(m)
)

FOREACH (actor IN split(row.actors, '|') |
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m)
)

FOREACH (genre IN split(row.genres, '|') |
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g)
)
"""

In [7]:
# ============================================================
# 7. LOAD MOVIE DATA INTO NEO4J
# ============================================================

graph.query(movie_query)

print("Movie dataset loaded successfully.")

Movie dataset loaded successfully.


In [8]:
# ============================================================
# 8. REFRESH NEO4J SCHEMA
# ============================================================

graph.refresh_schema()

print(graph.schema)

Node properties:
Person {age: INTEGER, name: STRING}
Company {name: STRING}
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:WORKS_AT]->(:Company)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [9]:
# ============================================================
# 9. CREATE GROQ LLM
# ============================================================

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=GROQ_MODEL,
    temperature=0,
)

print("Groq LLM initialized successfully.")

Groq LLM initialized successfully.


In [10]:
# ============================================================
# 10. TEST GROQ
# ============================================================

response = llm.invoke(
    "Explain Neo4j in one sentence."
)

print(response.content)

Neo4j is a high‑performance, ACID‑compliant graph database that stores and queries data as interconnected nodes and relationships, enabling intuitive modeling and fast traversal of complex networks.


In [12]:
# ============================================================
# 11. CREATE GRAPH CYPHER QA CHAIN
# ============================================================

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
    return_intermediate_steps=True,
)

print("GraphCypherQAChain created successfully.")

GraphCypherQAChain created successfully.


In [14]:
# ============================================================
# 12. QUESTION 1
# ============================================================

question = "Who was the director of the movie Casino?"

response = chain.invoke({
    "query": question
})

print("Question:")
print(question)

print("\nAnswer:")
print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'Casino'})
RETURN p.name AS director
Full Context:
[{'director': 'Martin Scorsese'}]

> Finished chain.
Question:
Who was the director of the movie Casino?

Answer:
Martin Scorsese.


In [15]:
# ============================================================
# 13. VIEW INTERMEDIATE STEPS
# ============================================================

for step in response["intermediate_steps"]:
    print(step)

{'query': "MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'Casino'})\nRETURN p.name AS director"}
{'context': [{'director': 'Martin Scorsese'}]}


In [16]:
# ============================================================
# 14. QUESTION 2
# ============================================================

question = "Who were the actors of the movie Casino?"

response = chain.invoke({
    "query": question
})

print("Question:")
print(question)

print("\nAnswer:")
print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: "Casino"})
RETURN p.name AS actor
Full Context:
[{'actor': 'Robert De Niro'}, {'actor': 'Joe Pesci'}, {'actor': 'Sharon Stone'}, {'actor': 'James Woods'}]

> Finished chain.
Question:
Who were the actors of the movie Casino?

Answer:
Robert De Niro, Joe Pesci, Sharon Stone, James Woods were the actors of the movie Casino.


In [17]:
# ============================================================
# 15. QUESTION 3
# ============================================================

question = "How many persons are in the database?"

response = chain.invoke({
    "query": question
})

print("Question:")
print(question)

print("\nAnswer:")
print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person) RETURN count(p) AS personCount;
Full Context:
[{'personCount': 1245}]

> Finished chain.
Question:
How many persons are in the database?

Answer:
1245 persons are in the database.


In [18]:
# ============================================================
# 16. QUESTION 4
# ============================================================

question = "How many movies has Tom Hanks acted in?"

response = chain.invoke({
    "query": question
})

print("Question:")
print(question)

print("\nAnswer:")
print(response["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: "Tom Hanks"})-[:ACTED_IN]->(m:Movie)
RETURN count(m) AS movieCount;
Full Context:
[{'movieCount': 2}]

> Finished chain.
Question:
How many movies has Tom Hanks acted in?

Answer:
Tom Hanks has acted in 2 movies.


In [19]:
# ============================================================
# 17. REUSABLE QUESTION FUNCTION
# ============================================================

def ask_question(question):
    
    response = chain.invoke({
        "query": question
    })

    print("\n" + "=" * 60)
    print("QUESTION")
    print("=" * 60)
    print(question)

    print("\n" + "=" * 60)
    print("ANSWER")
    print("=" * 60)
    print(response["result"])

    return response